In [1]:
%pip install python-binance pyspark

     |████████████████████████████████| 80 kB 1.4 MB/s            
     |████████████████████████████████| 1.2 MB 4.5 MB/s            
     |████████████████████████████████| 2.3 MB 4.1 MB/s            
     |████████████████████████████████| 53 kB 836 kB/s            
     |████████████████████████████████| 294 kB 3.5 MB/s            
     |████████████████████████████████| 163 kB 4.4 MB/s            
     |████████████████████████████████| 242 kB 4.0 MB/s            
     |████████████████████████████████| 320 kB 4.3 MB/s            
     |████████████████████████████████| 124 kB 4.5 MB/s            
     |████████████████████████████████| 781 kB 3.5 MB/s            
     |████████████████████████████████| 211 kB 3.0 MB/s            
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 3.10.0.2
    Uninstalling typing-extensions-3.10.0.2:
      Successfully uninstalled typing-extensions-3.10.0.2
Note: you may need to restart the kernel to use u

In [2]:
from binance.client import Client
import configparser
import iceberg_spark
from pyspark.sql.functions import *
import time
spark, jvm, hive_uri, sc = iceberg_spark.start_spark('s3', 'aws', 'hadoop')

# Read config file
config = configparser.ConfigParser()
config.read('config.ini')
api_secret = config['binance']['secret']
api_key = config['binance']['key']

# create client
client = Client(api_key, api_secret, tld='us')


Using S3 with Hive or Hadoop


:: loading settings :: url = jar:file:/usr/local/spark-3.1.2-bin-hadoop3.2/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark3-runtime added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c43d3590-524a-4754-ae94-ceaa43f2e765;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark3-runtime;0.12.1 in central
	found org.apache.hadoop#hadoop-aws;3.2.0 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.375 in central
:: resolution report :: resolve 539ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.375 from central in [default]
	org.apache.hadoop#hadoop-aws;3.2.0 from central in [default]
	org.apache.iceberg#iceberg-spark3-runtime;0.12.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            |

KeyError: 'binance'

In [ ]:
# get current prices from binance
tickers = client.get_all_tickers()

#convert to a spark dataframe, cast data type to decimal, and add date and timestamp columns
tickersDF = spark.createDataFrame(data=tickers)
tickersDF = tickersDF.withColumn("price", col("price").cast("decimal(38,8)"))
tickersDF = tickersDF.withColumn("date", current_date()).withColumn("ts", current_timestamp())


#write to an iceberg table
tickersDF.writeTo('tickers').using('iceberg').partitionedBy('date').createOrReplace()

In [ ]:
spark.table("tickers").show()

In [ ]:
# create funtion to append data
def get_tickers():
    # get current prices from binance
    tickers = client.get_all_tickers()
    
    #convert to a spark dataframe & add date and timestamp columns
    tickersDF = spark.createDataFrame(data=tickers)
    tickersDF = tickersDF.withColumn("price", col("price").cast("decimal(38,8)"))
    tickersDF = tickersDF.withColumn("date", current_date()).withColumn("ts", current_timestamp())
    
    
    #write to an iceberg table
    tickersDF.writeTo('tickers').append()

In [ ]:
# Function to repeat a task
def load_tickers(interval):
    while True:
        get_tickers()
        time.sleep(interval)

In [ ]:
# call periodic work every 10 seconds
load_tickers(10)

# Spark Writes
(https://iceberg.apache.org/spark-writes/) [https://iceberg.apache.org/spark-writes/]

# Writing with DataFrames
[apache Iceberg](https://iceberg.apache.org/spark-writes/#writing-with-dataframes)
<ul>
    <li>df.writeTo(t).create() is equivalent to CREATE TABLE AS SELECT</li>
    <li>df.writeTo(t).replace() is equivalent to REPLACE TABLE AS SELECT</li>
    <li>df.writeTo(t).append() is equivalent to INSERT INTO</li>
    <li>df.writeTo(t).overwritePartitions() is equivalent to dynamic INSERT OVERWRITE</li>
</ul>
